# Week 8 Final Exercise

## Capstone Project - Multi-Agent Code Generation Worflow

### Ruby on Rails Specialists Agents

This notebook configures 5 agents, all 5 are Ruby on Rails experts. Their roles are:
* System Analyst Agent
* Backend Developer Agent
* Frontend Developer Agent
* QA Tester Agent
* Plain Text to JSON Agent

First, System Analyst takes the user query and transforms it into an Requirements Document.
The document contains the details instructor for the Backend Developer, Frontend Developer,
and QA Tester Agent. When the three developer agents have finished generating the code then
the last agent Plain Text to JSON Agent will convert the outputs to structured JSON
with the fields `file` that is the filename, and `text` that is the code.

We will use LiteLLM framework to build the Agents.


In [ ]:
# Imports and environment setup
import os
from dotenv import load_dotenv
from litellm import completion
from pydantic import BaseModel
from system_analyst_agent import SystemAnalystAgent
from backend_developer_agent import BackendDeveloperAgent
from frontend_developer_agent import FrontendDeveloperAgent
from qa_tester_agent import QaTesterAgent

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
MODEL = 'gpt-5-mini'

# Agents setup
system_analyst_agent = SystemAnalystAgent()
backend_developer_agent = BackendDeveloperAgent()
frontend_developer_agent = FrontendDeveloperAgent()
qa_tester_agent = QaTesterAgent()


In [ ]:
# LLM output to code files JSON generator
class File(BaseModel):
    file: str
    text: str

class Files(BaseModel):
    files: list[File]

def make_prompt(document):
    return f"""
        Here is the files with their code content:
        {document}
        You respond in structured JSON with fields:
        - file: filename
        - text: code content
    """

def make_messages(document: str):
    return [
        {"role": "system", "content": """
            You convert the provided files names and code content to JSON.
            You respond in structured JSON with fields:
            - file: filename
            - text: code content"""},
        {"role": "user", "content": make_prompt(document)},
    ]

def generate_files(agent_response: str):
    messages = make_messages(agent_response)
    response = completion(model=MODEL, messages=messages, response_format=Files)
    reply = response.choices[0].message.content
    return Files.model_validate_json(reply).files


In [ ]:
# Chat function
def chat_with_agents(user_query: str):
    print(user_query)

    # Step 1: System Analyst Agent generates requirements document
    requirements_document = system_analyst_agent.run_agent(user_query)
    
    # Step 2: Backend Developer Agent generates backend code files
    backend_response = backend_developer_agent.run_agent(requirements_document)
    backend_files = generate_files(backend_response)
    
    # Step 3: Frontend Developer Agent generates frontend code files
    frontend_response = frontend_developer_agent.run_agent(requirements_document)
    frontend_files = generate_files(frontend_response)
    
    # Step 4: QA Tester Agent generates test cases and QA verification steps
    qa_response = qa_tester_agent.run_agent(requirements_document)
    qa_files = generate_files(qa_response)
    
    return {
        "requirements": requirements_document,
        "backend_files": backend_files,
        "frontend_files": frontend_files,
        "qa_files": qa_files,
        # "backend_files": [],
        # "frontend_files": [],
        # "qa_files": [],
    }


In [ ]:
# Gradio Interface with 4 sections for each agent's output
import gradio as gr

def gradio_interface(user_query):
    results = chat_with_agents(user_query)
    return results["requirements"], results["backend_files"], results["frontend_files"], results["qa_files"]

with gr.Blocks() as demo:
    gr.Markdown("## LLM Agent Collaboration Demo")
    user_query = gr.Textbox(label="User Query", placeholder="Describe the application you want to build...")
    submit_button = gr.Button("Submit")
    
    with gr.Tab("Requirements Document"):
        requirements_output = gr.Textbox(label="Generated Requirements Document", lines=10)
    
    with gr.Tab("Backend Code Files"):
        backend_output = gr.Dataframe(headers=["Filename", "Code Content"])
    
    with gr.Tab("Frontend Code Files"):
        frontend_output = gr.Dataframe(headers=["Filename", "Code Content"])
    
    with gr.Tab("QA Test Cases"):
        qa_output = gr.Dataframe(headers=["Filename", "Test Case Content"])
    
    submit_button.click(
        fn=gradio_interface, 
        inputs=user_query, 
        outputs=[requirements_output, backend_output, frontend_output, qa_output],
    )

# Launch interface in browser
demo.launch(inbrowser=True)

# Examples query:
# Create an Online Shop with Products, Categories, Orders, Custemers including a REST API.

